# Lenguajes y Autómatas II (SCD-1016)
## Sesión 30 · Hito 3: Generador de TAC y Optimizador End-to-End

**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`) · **Repositorio maestro:** https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2

---

> **Pregunta guía:** ¿Qué tan pequeño y correcto puede quedar el programa intermedio antes de convertirlo en ensamblador?
>
> **Objetivo de la sesión:** Integrar la traducción de un AST a TAC con el plegado de constantes, devolviendo el lugar donde queda el resultado y las instrucciones que sí hacen falta.

**Entregable:** tu función generar_tac_opt() en artesano_sesion30_c.c, el harness en verde y la bitácora en auditorias/auditoria_sesion30.md

**Tiempo:** 60 min · **Fases:** Artesano (manual) → Científico (IA con contrato) → Auditor (harness)

> ¿Dudas de C? Consulta la [guía de C](https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2/blob/main/docs/guia-c.md) antes de empezar.


## 1. ¿Qué vas a construir?

Recibes un árbol con hojas numéricas, variables y operadores binarios. Debes emitir solo las instrucciones necesarias: si un subárbol es constante, se calcula en compilación y no genera línea. Además, escribe en resultado el literal o el temporal que contiene el valor final.

Al final de la sesión tendrás: **Una función en C que traduce un AST a TAC optimizado, pliega constantes y reporta el lugar del resultado.**

Los conceptos y el porqué de esta sesión están en la presentación
([sesión 30](https://ijmarquezl.github.io/tecnm-lenguajes-automatas-2/presentaciones/sesion30.html)). Aquí solo está el trabajo: qué se
recibe, qué debe salir y cómo se comprueba.


## 2. El contrato exacto

El archivo `api_sesion30.h` declara la interfaz que el harness sabe probar.
Tu implementación **debe** respetar esas firmas; si las cambias, el harness no compila.

```c
#include "api_sesion30.h"

int generar_tac_opt(const Nodo *raiz, char lineas[][64], int max, char *resultado, int cap);
```

- Un subárbol con dos hojas numéricas se pliega a un literal y no emite instrucciones.
- Las identidades x + 0, x * 1 y x * 0 se resuelven sin generar una operación.
- No pliegues una división entre cero: deja que la instrucción llegue al backend.

> El harness compila con `-Wall -Wextra -Werror`: una advertencia del compilador cuenta como error.
>
> El harness compila `harness_sesion30_c.c` junto con `artesano_sesion30_c.c`
> (o `ia_sesion30_c.c`). Por eso tu archivo **no debe tener `main()`**: el `main()`
> es del harness.


## 3. Cómo se califica

### Casos que evalúa el harness

| # | Caso | Resultado esperado |
| :-: | :--- | :--- |
| 1 | `x + 2 * 3 (linea)` | `_t0 = x + 6` |
| 2 | `x + 2 * 3 (resultado)` | `_t0` |
| 3 | `x * y + 2 * 3` | `_t1 = _t0 + 6` |
| 4 | `todo constante: sin lineas` | 0 |
| 5 | `todo constante: resultado` | `45` |
| 6 | `x + 0 (identidad)` | `x` |

### Salida literal que debes obtener

```text
=== AUDITORIA SESION 30 - AST a TAC optimizado de punta a punta ===
Caso                               esperado:              obtenido:              resultado
[ 1] x + 2 * 3 (linea)                  esperado: _t0 = x + 6  obtenido: _t0 = x + 6  OK
[ 2] x + 2 * 3 (resultado)              esperado: _t0          obtenido: _t0          OK
[ 3] x * y + 2 * 3
     esperado: _t1 = _t0 + 6
     obtenido: _t1 = _t0 + 6
     -> OK
[ 4] todo constante: sin lineas         esperado: 0            obtenido: 0            OK
[ 5] todo constante: resultado          esperado: 45           obtenido: 45           OK
[ 6] x + 0 (identidad)                  esperado: x            obtenido: x            OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion30.md
```

Los números entre corchetes son el contador de casos. Mientras algo falle verás
`FALLA` en esa línea, el total será menor y el programa terminará con código de
salida 1.

### Criterios de aceptación

- [ ] Un subárbol constante se sustituye por su valor y no genera instrucciones.
- [ ] El lugar del resultado coincide con el último temporal emitido o con el literal plegado.
- [ ] El TAC optimizado conserva exactamente el valor que produciría el árbol original.


## Paso 0 · Entorno


In [1]:
import os

if os.system('which gcc > /dev/null 2>&1') != 0:
    print('⏳ Instalando compilador...')
    !apt-get update -qq && apt-get install -y -qq gcc > /dev/null 2>&1

print('✅ Entorno listo:')
!gcc --version | head -n 1


✅ Entorno listo:
gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0


## Fase 1 · El Artesano (15 min, sin IA)

Escribe a mano la implementación en `artesano_sesion30_c.c`. El archivo ya trae la
firma y un `TODO`: edita el contenido de la celda `%%writefile` que está abajo (o el
archivo en el panel **Archivos** de Colab) y vuelve a ejecutarla para guardar los
cambios. Hazlo sin consultar a la IA. Si te atoras, anota en tu bitácora exactamente
en qué te atoraste; eso también es evidencia.


In [2]:
%%writefile api_sesion30.h
/* api_sesion30.h - contrato de la Sesion 30
 * AST a TAC optimizado de punta a punta
 *
 * Esta cabecera es el contrato: el harness y tu implementacion
 * deben incluirla. Si cambias una firma, el harness dejara de compilar.
 */
#ifndef API_SESION30_H
#define API_SESION30_H

typedef struct Nodo {
    char tipo;
    char op;
    char valor[16];
    struct Nodo *izq;
    struct Nodo *der;
} Nodo;

int generar_tac_opt(const Nodo *raiz, char lineas[][64], int max, char *resultado, int cap);

#endif /* API_SESION30_H */


Writing api_sesion30.h


In [10]:
%%writefile artesano_sesion30_c.c
#include "api_sesion30.h"
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <limits.h>

typedef struct {
    int constante;
    long valor;
    char texto[64];
} Valor;

static int copiar_texto(char *destino, size_t capacidad,
                        const char *origen) {
    int n;

    if (destino == NULL || origen == NULL || capacidad == 0)
        return 0;

    n = snprintf(destino, capacidad, "%s", origen);

    return n >= 0 && (size_t)n < capacidad;
}

static int convertir_numero(const char *texto, long *valor) {
    char *fin;
    long numero;

    if (texto == NULL || texto[0] == '\0')
        return 0;

    numero = strtol(texto, &fin, 10);

    if (*fin != '\0')
        return 0;

    *valor = numero;
    return 1;
}

static int calcular(char op, long a, long b, long *r) {
    switch (op) {
        case '+':
            if ((b > 0 && a > LONG_MAX - b) ||
                (b < 0 && a < LONG_MIN - b))
                return 0;
            *r = a + b;
            return 1;

        case '-':
            if ((b < 0 && a > LONG_MAX + b) ||
                (b > 0 && a < LONG_MIN + b))
                return 0;
            *r = a - b;
            return 1;

        case '*':
            if (a != 0 && b != 0) {
                if ((a == -1 && b == LONG_MIN) ||
                    (b == -1 && a == LONG_MIN))
                    return 0;

                if (a > 0 && b > 0 && a > LONG_MAX / b)
                    return 0;

                if (a > 0 && b < 0 && b < LONG_MIN / a)
                    return 0;

                if (a < 0 && b > 0 && a < LONG_MIN / b)
                    return 0;

                if (a < 0 && b < 0 && a < LONG_MAX / b)
                    return 0;
            }
            *r = a * b;
            return 1;

        case '/':
            if (b == 0 || (a == LONG_MIN && b == -1))
                return 0;
            *r = a / b;
            return 1;

        default:
            return 0;
    }
}

static int generar(const Nodo *n, char lineas[][64],
                   int max, int *contador, Valor *salida) {
    Valor izquierda;
    Valor derecha;
    long resultado;
    char temporal[64];
    int nchar;

    if (n == NULL || salida == NULL)
        return 0;

    /* Hoja numerica */
    if (n->tipo == 'N') {
        if (!convertir_numero(n->valor, &salida->valor))
            return 0;

        salida->constante = 1;

        return copiar_texto(salida->texto,
                            sizeof(salida->texto), n->valor);
    }

    /* Hoja variable */
    if (n->tipo == 'V') {
        salida->constante = 0;

        return copiar_texto(salida->texto,
                            sizeof(salida->texto), n->valor);
    }

    /* Operador binario */
    if (n->tipo != 'B')
        return 0;

    if (!generar(n->izq, lineas, max, contador, &izquierda))
        return 0;

    if (!generar(n->der, lineas, max, contador, &derecha))
        return 0;

    /* Plegado de constantes */
    if (izquierda.constante && derecha.constante &&
        calcular(n->op, izquierda.valor,
                 derecha.valor, &resultado)) {

        salida->constante = 1;
        salida->valor = resultado;

        nchar = snprintf(salida->texto,
                         sizeof(salida->texto), "%ld", resultado);

        return nchar >= 0 &&
               (size_t)nchar < sizeof(salida->texto);
    }

    /* x + 0 */
    if (n->op == '+' && izquierda.constante &&
        izquierda.valor == 0) {
        *salida = derecha;
        return 1;
    }

    if (n->op == '+' && derecha.constante &&
        derecha.valor == 0) {
        *salida = izquierda;
        return 1;
    }

    /* x * 1 */
    if (n->op == '*' && izquierda.constante &&
        izquierda.valor == 1) {
        *salida = derecha;
        return 1;
    }

    if (n->op == '*' && derecha.constante &&
        derecha.valor == 1) {
        *salida = izquierda;
        return 1;
    }

    /* x * 0 */
    if (n->op == '*' &&
        ((izquierda.constante && izquierda.valor == 0) ||
         (derecha.constante && derecha.valor == 0))) {

        salida->constante = 1;
        salida->valor = 0;

        return copiar_texto(salida->texto,
                            sizeof(salida->texto), "0");
    }

    /* Crear temporal */
    if (*contador >= max)
        return 0;

    nchar = snprintf(temporal, sizeof(temporal),
                     "_t%d", *contador);

    if (nchar < 0 || (size_t)nchar >= sizeof(temporal))
        return 0;

    nchar = snprintf(lineas[*contador], 64,
                     "%s = %s %c %s",
                     temporal,
                     izquierda.texto,
                     n->op,
                     derecha.texto);

    if (nchar < 0 || nchar >= 64)
        return 0;

    (*contador)++;

    salida->constante = 0;

    return copiar_texto(salida->texto,
                        sizeof(salida->texto), temporal);
}

int generar_tac_opt(const Nodo *raiz, char lineas[][64],
                    int max, char *resultado, int cap) {
    Valor final;
    int contador = 0;

    if (raiz == NULL || lineas == NULL ||
        resultado == NULL || max < 0 || cap <= 0)
        return -1;

    resultado[0] = '\0';

    if (!generar(raiz, lineas, max, &contador, &final))
        return -1;

    if (!copiar_texto(resultado, (size_t)cap, final.texto))
        return -1;

    return contador;
}



Overwriting artesano_sesion30_c.c


## Fase 2 · El Científico (10 min)

Redacta el contrato **C-R-E-O** (Contexto, Rol, Especificación, Output) para pedirle
la misma función a la IA, y pega el resultado en la celda de `ia_sesion30_c.c`. La
firma debe ser idéntica a la del contrato; si la IA la cambia, corrígela antes de
compilar.


In [4]:
%%writefile ia_sesion30_c.c
#include "api_sesion30.h"
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <limits.h>

typedef struct {
    int es_constante;
    long numero;
    char nombre[32];
} Resultado;

static int guardar(char *destino, size_t capacidad,
                   const char *origen) {
    int n = snprintf(destino, capacidad, "%s", origen);

    return n >= 0 && (size_t)n < capacidad;
}

static int leer_numero(const char *texto, long *numero) {
    char *fin;
    long valor;

    if (texto == NULL || texto[0] == '\0')
        return 0;

    valor = strtol(texto, &fin, 10);

    if (*fin != '\0')
        return 0;

    *numero = valor;
    return 1;
}

static int operar(char operador, long a, long b, long *r) {
    switch (operador) {
        case '+':
            if ((b > 0 && a > LONG_MAX - b) ||
                (b < 0 && a < LONG_MIN - b))
                return 0;
            *r = a + b;
            return 1;

        case '-':
            if ((b < 0 && a > LONG_MAX + b) ||
                (b > 0 && a < LONG_MIN + b))
                return 0;
            *r = a - b;
            return 1;

        case '*':
            if (a != 0 && b != 0) {
                if ((a == -1 && b == LONG_MIN) ||
                    (b == -1 && a == LONG_MIN))
                    return 0;

                if (a > 0 && b > 0 && a > LONG_MAX / b)
                    return 0;

                if (a > 0 && b < 0 && b < LONG_MIN / a)
                    return 0;

                if (a < 0 && b > 0 && a < LONG_MIN / b)
                    return 0;

                if (a < 0 && b < 0 && a < LONG_MAX / b)
                    return 0;
            }

            *r = a * b;
            return 1;

        case '/':
            if (b == 0 || (a == LONG_MIN && b == -1))
                return 0;

            *r = a / b;
            return 1;

        default:
            return 0;
    }
}

static int generar(const Nodo *n, char lineas[][64],
                   int max, int *total, Resultado *r) {
    Resultado izquierda;
    Resultado derecha;
    long valor;
    char temporal[32];
    int nchar;

    if (n == NULL || r == NULL)
        return 0;

    /* Nodo numerico */
    if (n->tipo == 'N') {
        if (!leer_numero(n->valor, &r->numero))
            return 0;

        r->es_constante = 1;
        return guardar(r->nombre, sizeof(r->nombre), n->valor);
    }

    /* Nodo variable */
    if (n->tipo == 'V') {
        r->es_constante = 0;
        return guardar(r->nombre, sizeof(r->nombre), n->valor);
    }

    /* Nodo de operacion binaria */
    if (n->tipo != 'B')
        return 0;

    if (!generar(n->izq, lineas, max, total, &izquierda))
        return 0;

    if (!generar(n->der, lineas, max, total, &derecha))
        return 0;

    /* Constant Folding */
    if (izquierda.es_constante && derecha.es_constante &&
        operar(n->op, izquierda.numero, derecha.numero, &valor)) {

        r->es_constante = 1;
        r->numero = valor;

        nchar = snprintf(r->nombre, sizeof(r->nombre),
                         "%ld", valor);

        return nchar >= 0 &&
               (size_t)nchar < sizeof(r->nombre);
    }

    /* Identidades: x + 0 */
    if (n->op == '+' && izquierda.es_constante &&
        izquierda.numero == 0) {
        *r = derecha;
        return 1;
    }

    if (n->op == '+' && derecha.es_constante &&
        derecha.numero == 0) {
        *r = izquierda;
        return 1;
    }

    /* Identidades: x * 1 */
    if (n->op == '*' && izquierda.es_constante &&
        izquierda.numero == 1) {
        *r = derecha;
        return 1;
    }

    if (n->op == '*' && derecha.es_constante &&
        derecha.numero == 1) {
        *r = izquierda;
        return 1;
    }

    /* Identidad: x * 0 */
    if (n->op == '*' &&
        ((izquierda.es_constante && izquierda.numero == 0) ||
         (derecha.es_constante && derecha.numero == 0))) {

        r->es_constante = 1;
        r->numero = 0;

        return guardar(r->nombre, sizeof(r->nombre), "0");
    }

    /* Generacion de TAC */
    if (*total >= max)
        return 0;

    nchar = snprintf(temporal, sizeof(temporal),
                     "_t%d", *total);

    if (nchar < 0 || (size_t)nchar >= sizeof(temporal))
        return 0;

    nchar = snprintf(lineas[*total], 64,
                     "%s = %s %c %s",
                     temporal,
                     izquierda.nombre,
                     n->op,
                     derecha.nombre);

    if (nchar < 0 || nchar >= 64)
        return 0;

    (*total)++;

    r->es_constante = 0;

    return guardar(r->nombre, sizeof(r->nombre), temporal);
}

int generar_tac_opt(const Nodo *raiz, char lineas[][64],
                    int max, char *resultado, int cap) {
    Resultado final;
    int instrucciones = 0;

    if (raiz == NULL || lineas == NULL ||
        resultado == NULL || max < 0 || cap <= 0)
        return -1;

    resultado[0] = '\0';

    if (!generar(raiz, lineas, max,
                 &instrucciones, &final))
        return -1;

    if (!guardar(resultado, (size_t)cap, final.nombre))
        return -1;

    return instrucciones;
}


Writing ia_sesion30_c.c


## Fase 3 · El Auditor (10 min)

El harness de abajo **no aprueba nada por sí solo**: ejecuta tu función caso por caso,
la compara contra el valor esperado e imprime `OK` o `FALLA`. Léelo antes de ejecutarlo:
los casos te dicen qué se está midiendo.


In [5]:
%%writefile harness_sesion30_c.c
/* harness_sesion30_c.c - auditoria automatica de la Sesion 30
 *
 * Compilacion:
 *   gcc -std=c11 -Wall -Wextra -Werror harness_sesion30_c.c artesano_sesion30_c.c -o test_artesano
 *
 * El harness NO aprueba nada por si solo: cada linea se calcula
 * ejecutando tu implementacion y comparandola contra el valor esperado.
 * Si algun caso falla, el programa termina con codigo de salida 1.
 */
#include <stdio.h>
#include <string.h>
#include "api_sesion30.h"

static int probadas = 0;
static int fallidas = 0;

static void verificar_str(const char *caso, const char *esperado,
                          const char *obtenido) {
    int ok = (obtenido != NULL) && (strcmp(esperado, obtenido) == 0);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12s obtenido: %-12s %s\n",
           probadas, caso, esperado, obtenido ? obtenido : "(NULL)",
           ok ? "OK" : "FALLA");
}

static void verificar_str_largo(const char *caso, const char *esperado,
                                const char *obtenido) {
    int ok = (obtenido != NULL) && (strcmp(esperado, obtenido) == 0);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %s\n", probadas, caso);
    printf("     esperado: %s\n", esperado);
    printf("     obtenido: %s\n", obtenido ? obtenido : "(NULL)");
    printf("     -> %s\n", ok ? "OK" : "FALLA");
}

static void verificar_int(const char *caso, long esperado, long obtenido) {
    int ok = (esperado == obtenido);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12ld obtenido: %-12ld %s\n",
           probadas, caso, esperado, obtenido, ok ? "OK" : "FALLA");
}

int main(void) {
    printf("=== AUDITORIA SESION 30 - AST a TAC optimizado de punta a punta ===\n");
    printf("Caso                               esperado:              obtenido:              resultado\n");

    { Nodo x = {.tipo='V', .valor="x"}; Nodo n2 = {.tipo='N', .valor="2"}; Nodo n3 = {.tipo='N', .valor="3"}; Nodo m = {.tipo='B', .op='*', .izq=&n2, .der=&n3}; Nodo s = {.tipo='B', .op='+', .izq=&x, .der=&m}; char l[8][64] = {{0}}; char r[16] = ""; generar_tac_opt(&s, l, 8, r, 16); verificar_str("x + 2 * 3 (linea)", "_t0 = x + 6", l[0]); }
    { Nodo x = {.tipo='V', .valor="x"}; Nodo n2 = {.tipo='N', .valor="2"}; Nodo n3 = {.tipo='N', .valor="3"}; Nodo m = {.tipo='B', .op='*', .izq=&n2, .der=&n3}; Nodo s = {.tipo='B', .op='+', .izq=&x, .der=&m}; char l[8][64] = {{0}}; char r[16] = ""; generar_tac_opt(&s, l, 8, r, 16); verificar_str("x + 2 * 3 (resultado)", "_t0", r); }
    { Nodo x = {.tipo='V', .valor="x"}; Nodo y = {.tipo='V', .valor="y"}; Nodo n2 = {.tipo='N', .valor="2"}; Nodo n3 = {.tipo='N', .valor="3"}; Nodo p = {.tipo='B', .op='*', .izq=&x, .der=&y}; Nodo m = {.tipo='B', .op='*', .izq=&n2, .der=&n3}; Nodo s = {.tipo='B', .op='+', .izq=&p, .der=&m}; char l[8][64] = {{0}}; char r[16] = ""; generar_tac_opt(&s, l, 8, r, 16); verificar_str_largo("x * y + 2 * 3", "_t1 = _t0 + 6", l[1]); }
    { Nodo n2 = {.tipo='N', .valor="2"}; Nodo n3 = {.tipo='N', .valor="3"}; Nodo n4 = {.tipo='N', .valor="4"}; Nodo n5 = {.tipo='N', .valor="5"}; Nodo s1 = {.tipo='B', .op='+', .izq=&n2, .der=&n3}; Nodo s2 = {.tipo='B', .op='+', .izq=&n4, .der=&n5}; Nodo m = {.tipo='B', .op='*', .izq=&s1, .der=&s2}; char l[8][64] = {{0}}; char r[16] = ""; int n = generar_tac_opt(&m, l, 8, r, 16); verificar_int("todo constante: sin lineas", (long)(0), (long)(n)); }
    { Nodo n2 = {.tipo='N', .valor="2"}; Nodo n3 = {.tipo='N', .valor="3"}; Nodo n4 = {.tipo='N', .valor="4"}; Nodo n5 = {.tipo='N', .valor="5"}; Nodo s1 = {.tipo='B', .op='+', .izq=&n2, .der=&n3}; Nodo s2 = {.tipo='B', .op='+', .izq=&n4, .der=&n5}; Nodo m = {.tipo='B', .op='*', .izq=&s1, .der=&s2}; char l[8][64] = {{0}}; char r[16] = ""; generar_tac_opt(&m, l, 8, r, 16); verificar_str("todo constante: resultado", "45", r); }
    { Nodo x = {.tipo='V', .valor="x"}; Nodo z = {.tipo='N', .valor="0"}; Nodo s = {.tipo='B', .op='+', .izq=&x, .der=&z}; char l[4][64] = {{0}}; char r[16] = ""; generar_tac_opt(&s, l, 4, r, 16); verificar_str("x + 0 (identidad)", "x", r); }

    printf("\nRESULTADO: %d/%d pruebas superadas\n",
           probadas - fallidas, probadas);
    if (fallidas != 0) {
        printf("Aun hay %d caso(s) sin resolver. Revisa tu implementacion.\n",
               fallidas);
        return 1;
    }
    printf("TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion30.md" "\n");
    return 0;
}


Writing harness_sesion30_c.c


In [11]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion30_c.c artesano_sesion30_c.c -o test_artesano && ./test_artesano

=== AUDITORIA SESION 30 - AST a TAC optimizado de punta a punta ===
Caso                               esperado:              obtenido:              resultado
[ 1] x + 2 * 3 (linea)                  esperado: _t0 = x + 6  obtenido: _t0 = x + 6  OK
[ 2] x + 2 * 3 (resultado)              esperado: _t0          obtenido: _t0          OK
[ 3] x * y + 2 * 3
     esperado: _t1 = _t0 + 6
     obtenido: _t1 = _t0 + 6
     -> OK
[ 4] todo constante: sin lineas         esperado: 0            obtenido: 0            OK
[ 5] todo constante: resultado          esperado: 45           obtenido: 45           OK
[ 6] x + 0 (identidad)                  esperado: x            obtenido: x            OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion30.md


In [12]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion30_c.c ia_sesion30_c.c -o test_ia && ./test_ia

=== AUDITORIA SESION 30 - AST a TAC optimizado de punta a punta ===
Caso                               esperado:              obtenido:              resultado
[ 1] x + 2 * 3 (linea)                  esperado: _t0 = x + 6  obtenido: _t0 = x + 6  OK
[ 2] x + 2 * 3 (resultado)              esperado: _t0          obtenido: _t0          OK
[ 3] x * y + 2 * 3
     esperado: _t1 = _t0 + 6
     obtenido: _t1 = _t0 + 6
     -> OK
[ 4] todo constante: sin lineas         esperado: 0            obtenido: 0            OK
[ 5] todo constante: resultado          esperado: 45           obtenido: 45           OK
[ 6] x + 0 (identidad)                  esperado: x            obtenido: x            OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion30.md


## Resultado esperado

Si tu implementación es correcta, la última línea del harness será:

```text
RESULTADO: 6/6 pruebas superadas
```

y la celda terminará sin error (código de salida 0). Compara tu salida completa
contra la de la sección 3: debe ser idéntica, línea por línea.


## Entrega

1. Copia `plantillas/auditoria_sesion30.md` a `auditorias/auditoria_sesion30.md` en tu fork.
2. Adjunta la salida del harness (las 6 líneas con `OK`).
3. Explica en el reporte crítico qué caso borde omitió la IA y cómo lo detectaste.
4. Haz commit y sincroniza tu fork.

Este laboratorio alimenta el 40% de laboratorios y es la base del Hito correspondiente
del Proyecto Integrador.
